## Prerequisite

In [ ]:
%reload_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt

from qiskit_metal import designs
from qiskit_metal import MetalGUI

import os

In [ ]:
os.makedirs("results/one_qubit_chip", exist_ok=True)

# 1. Create the Qbit design

In [ ]:
design = designs.DesignPlanar({}, True)

design.overwrite_enabled=True

design.chips.main.size['size_x'] = '2mm'
design.chips.main.size['size_y'] = '2mm'

gui = MetalGUI(design)

In [ ]:
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket

design.delete_all_components()

q1 = TransmonPocket(design, 'Q1', options = dict(
    pad_width = '425 um', 
    pocket_height = '650um',
    connection_pads=dict(
        readout = dict(loc_W=+1,loc_H=+1, pad_width='200um')
    )))

gui.rebuild()
gui.autoscale()

# 2. Analyze the transmon using the Eigenmode method

In [ ]:
from qiskit_metal.analyses.quantization import EPRanalysis
eig_qb = EPRanalysis(design, "hfss")

In [ ]:
eig_qb.sim.setup

In [ ]:
eig_qb.sim.run(name="SingleQubit",  components=['Q1'], open_terminations=[])

In [ ]:
eig_qb.sim.plot_convergences()

# 3. Analyze the transmon using the EPR method

In [ ]:
eig_qb.del_junction()
eig_qb.add_junction('jj', 'Lj', 'Cj', rect='JJ_rect_Lj_Q1_rect_jj', line='JJ_Lj_Q1_rect_jj_')
eig_qb.setup.sweep_variable = 'Lj'
eig_qb.setup

In [ ]:
eig_qb.run_epr()

In [ ]:
from pyEPR.core_quantum_analysis import QuantumAnalysis

file_name='C:\\data-pyEPR\Project11\\SingleQubit_hfss\\2025-08-29 06-54-58.npz' # path to results file

qa = QuantumAnalysis(file_name)

result = qa.analyze_all_variations(cos_trunc = 8, fock_trunc = 7)
# qa.analyze_variation('0', cos_trunc=8, fock_trunc=7, print_result=False)

print((result))

In [ ]:
res = qa.results['0']
chi_nd_matrix = res['chi_ND']
print("chi_ND (MHz):\n", chi_nd_matrix)

In [ ]:
from qiskit_metal.analyses.hamiltonian.transmon_charge_basis import Hcpb

ω = list(result.items())[0][1]['f_0'][0]
α = chi_nd_matrix[0][0]
print(ω, α)

We will use the "params_from_spectrum" function to calculate the target $E_j$ and $E_c$ values for a extracted transmon frequency and anharmonicity.

In [ ]:
H = Hcpb()
EjEc = H.params_from_spectrum(ω, α)

print(EjEc)
print("transmon frequency:", H.fij(0,1), "anharmonicity:", H.anharm())

# 4. Analysis

## Energy levels, Charge Dispersion, Energy Level Differences, Anharmonicity and Dephasing Time (T2) 
### Based on the 4.34 Transmon qubit CPB hamiltonian charge basis notebook of the official Qiskit Metal

### Energy levels

In [ ]:
Ej, Ec = EjEc

In [ ]:
x = np.linspace(-2.0,2.0,101) # this represents the charging energy (ng)
H_norm = Hcpb(nlevels=2, Ej=Ej, Ec=Ec, ng=0.5) # Hamiltonian definition 
norm = H_norm.fij(0,1) # normalization constant

In [ ]:
E0, E1, E2 = [], [], []

# For a given value of offset charge (ng, represented by x) we will calculate the CPB Hamiltonian using the previously assigned values of E_J and E_C. Then we calculate the eigenvalue for a given value of m.
for i in x: 
    H = Hcpb(nlevels=3, Ej=Ej, Ec=Ec, ng=i)
    E0.append(H.evalue_k(0)/norm)
    E1.append(H.evalue_k(1)/norm)
    E2.append(H.evalue_k(2)/norm)

# define the minimum of E0 and set this to E=0
floor = min(E0) 
 
plt.plot(x, E0 - floor, 'k', label="m=0")
plt.plot(x, E1 - floor, 'r', label="m=1")
plt.plot(x, E2 - floor, 'b', label="m=2")
plt.xlabel("ng")
plt.ylabel("Em/E01")
plt.legend(title="Energy Level:", loc='upper right')

In [ ]:
plt.savefig("results/one_qubit_chip/energy_levels.png", dpi=300, bbox_inches='tight')
plt.close()

### Charge Dispersion

In [ ]:
E_c = Ec
epsilon0, epsilon1, epsilon2, epsilon3 = [], [], [], []    # charge dispersion for m=0 through m=4
x = np.linspace(1,140,101)           # this this ratio of Ej/Ec which will go on the x-axis. 

In [ ]:
for i in x:
    E_j = i*E_c 
    H_zero = Hcpb(nlevels=15, Ej=i*E_c, Ec=E_c, ng=0.0)
    H_half = Hcpb(nlevels=15, Ej=i*E_c, Ec=E_c, ng=0.5)
    
    H_norm = Hcpb(nlevels=15, Ej=i*E_c, Ec=E_c, ng=0.5)
    norm = H_norm.fij(0,1)                         # normalization constant 
    
    epsilon0.append(abs(H_half.evalue_k(0) - H_zero.evalue_k(0))/norm)
    epsilon1.append(abs(H_half.evalue_k(1) - H_zero.evalue_k(1))/norm)
    epsilon2.append(abs(H_half.evalue_k(2) - H_zero.evalue_k(2))/norm)
    epsilon3.append(abs(H_half.evalue_k(3) - H_zero.evalue_k(3))/norm)

In [ ]:
plt.plot(x, epsilon0, 'k', label="m=0")
plt.plot(x, epsilon1, 'b', label="m=1")
plt.plot(x, epsilon2, 'r', label="m=2")
plt.plot(x, epsilon3, 'g', label="m=3") 
plt.yscale("log")
plt.xlabel("EJ/Ec")
plt.ylabel("Epsilon/E01")
plt.legend(title="Energy Level", loc='upper right')

In [ ]:
plt.savefig("results/one_qubit_chip/charge_dispersion.png", dpi=300, bbox_inches='tight')
plt.close()

### Energy Level Differences

In [ ]:
E00, E10, E20, E30 = [], [], [], [] 
E0, E1, E2, E3 = [], [], [], []

for i in x:
    H = Hcpb(nlevels=15, Ej=i*E_c, Ec=E_c, ng=0.5)
    E0 = H.evalue_k(0) 
    E1 = H.evalue_k(1)
    E2 = H.evalue_k(2)
    E3 = H.evalue_k(3) 
    E00.append((E0 - E0)/E_c)
    E10.append((E1 - E0)/E_c)
    E20.append((E2 - E0)/E_c)
    E30.append((E3 - E0)/E_c)

plt.plot(x,E00,'k',label="m0")
plt.plot(x,E10,'b',label="m=1")
plt.plot(x,E20,'g',label="m=2")
plt.plot(x,E30,'r',label="m=3") 
plt.xlabel("Ej/Ec")
plt.ylabel("Em0/Ec")
plt.legend(title="Energy Levels", loc='upper right')

In [ ]:
plt.savefig("results/one_qubit_chip/energy_level_differences.png", dpi=300, bbox_inches='tight')
plt.close()

### Anharmonicity

In [ ]:
x = np.linspace(0,80,101)   #EJ/EC
alpha = [] 
alpha_r = [] 

In [ ]:
for i in x:
    H_anharm = Hcpb(nlevels=15, Ej=i*E_c, Ec=E_c, ng=0.5)
    alpha.append(H_anharm.anharm()) 
    alpha_r.append(H_anharm.anharm()/H_anharm.fij(0,1))

In [ ]:
plt.figure(1)
plt.subplot(131)
plt.plot(x,alpha)
plt.xlabel("Ej/Ec")
plt.ylabel("alpha") 
plt.subplot(133)
plt.plot(x,alpha_r)
plt.ylim(-0.2, 1.0) 
plt.xlabel("Ej/Ec")
plt.ylabel("alpha_r")

In [ ]:
plt.savefig("results/one_qubit_chip/anharmonicity.png", dpi=300, bbox_inches='tight')
plt.close()

### Dephasing Time (T2)

In [ ]:
x = np.linspace(0,80,101)     # ratio of Ej/Ec, varying from 0 to 80
T2 = []                       # empty list for T2 

In [ ]:
E_c = Ec
for i in x: 
    H_half = Hcpb(nlevels=15, Ej=i*E_c, Ec=E_c, ng=0.5)
    H_zero = Hcpb(nlevels=15, Ej=i*E_c, Ec=E_c, ng=0.0)
    eps = abs(H_half.evalue_k(1) - H_zero.evalue_k(1))    
    T2.append(1.0/(2.0*(1E-4)*(1E6)*eps))

plt.plot(x, T2)
plt.yscale("log")
plt.xlabel("Ej/Ec")
plt.ylabel("T2 (sec)")

In [ ]:
plt.savefig("results/one_qubit_chip/dephasing_time.png", dpi=300, bbox_inches='tight')
plt.close()

# 5. Qiskit Pulse Schedules with Qiskit Dynamics

### https://qiskit-community.github.io/qiskit-dynamics/tutorials/qiskit_pulse.html?utm_source=chatgpt.com

In [ ]:
import numpy as np
import qiskit.pulse as pulse
# Strength of the Rabi-rate in GHz.
r = 0.1

# Frequency of the qubit transition in GHz.
w = ω 

# Sample rate of the backend in ns.
dt = 1 / 4.5

# Define gaussian envelope function to approximately implement an sx gate.
amp = 1. / 1.75
sig = 0.6985/r/amp
T = 4*sig
duration = int(T / dt)
beta = 2.0

with pulse.build(name="sx-sy schedule") as sxp:
    pulse.play(pulse.Drag(duration, amp, sig / dt, beta), pulse.DriveChannel(0))
    pulse.shift_phase(np.pi/2, pulse.DriveChannel(0))
    pulse.play(pulse.Drag(duration, amp, sig / dt, beta), pulse.DriveChannel(0))

sxp.draw()

In [ ]:
from matplotlib import pyplot as plt
from qiskit_dynamics.pulse import InstructionToSignals

plt.rcParams["font.size"] = 16

converter = InstructionToSignals(dt, carriers={"d0": w})

signals = converter.get_signals(sxp)
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, title in zip(axs, ["envelope", "signal"]):
    signals[0].draw(0, 2*T, 2000, title, axis=ax)
    ax.set_xlabel("Time (ns)")
    ax.set_ylabel("Amplitude")
    ax.set_title(title)
    ax.vlines(T, ax.get_ylim()[0], ax.get_ylim()[1], "k", linestyle="dashed")

In [ ]:
from qiskit.quantum_info.operators import Operator
from qiskit_dynamics import Solver

# construct operators
X = Operator.from_label('X').data
Z = Operator.from_label('Z').data

drift = 2 * np.pi * w * Z/2
operators = [2 * np.pi * r * X/2]

# construct the solver
hamiltonian_solver = Solver(
    static_hamiltonian=drift,
    hamiltonian_operators=operators,
    rotating_frame=drift,
    rwa_cutoff_freq=2 * 5.0,
    hamiltonian_channels=['d0'],
    channel_carrier_freqs={'d0': w},
    dt=dt
)

In [ ]:
from qiskit.quantum_info.states import Statevector

# Start the qubit in its ground state.
y0 = Statevector([1., 0.])

%time sol = hamiltonian_solver.solve(t_span=[0., 2*T], y0=y0, signals=sxp, atol=1e-8, rtol=1e-8)

In [ ]:
def plot_populations(sol):
    pop0 = [psi.probabilities()[0] for psi in sol.y]
    pop1 = [psi.probabilities()[1] for psi in sol.y]

    fig = plt.figure(figsize=(8, 5))
    plt.plot(sol.t, pop0, lw=3, label="Population in |0>")
    plt.plot(sol.t, pop1, lw=3, label="Population in |1>")
    plt.xlabel("Time (ns)")
    plt.ylabel("Population")
    plt.legend(frameon=False)
    plt.ylim([0, 1.05])
    plt.xlim([0, 2*T])
    plt.vlines(T, 0, 1.05, "k", linestyle="dashed")

In [ ]:
plot_populations(sol)

# 6. Estimate the T1, T2 Relaxation times using Qutip

### 6. Create the  Duffing Hamiltonian

In [ ]:
f_01 = result['0']['f_0'][0]            # in Hz
alpha = result['0']['chi_O1'][0][0] * 1e6     # in Hz
f_01, alpha

In [ ]:
from qutip import destroy
import numpy as np

w_q = 2 * np.pi * f_01
alpha_rad = 2 * np.pi * alpha

N = 5
a = destroy(N)

H = w_q * (a.dag() * a) + (alpha_rad / 2) * ((a.dag() * a.dag()) * (a * a))
H

In [ ]:
from qutip import *

tlist = np.linspace(0, 40e-6, 1000)  # seconds

opts = Options(nsteps=10000)

T1 = 25e-6  # seconds
gamma1 = 1 / T1
c_ops_T1 = [np.sqrt(gamma1) * a]

# Prepare the Initial State
psi_T1 = basis(N, 1)
result_T1 = mesolve(H, psi_T1, tlist, c_ops_T1, [a.dag()*a], options=opts)

# Collapse operators for T2 (T_phi included)
T_phi = 40e-6  # seconds
gamma_phi = 1 / T_phi
c_ops_T2 = [np.sqrt(gamma1) * a, np.sqrt(gamma_phi) * a.dag() * a]

# Initial state for T2 simulation: superposition (|0> + |1>)/sqrt(2)
psi_T2 = (basis(N, 0) + basis(N, 1)).unit()
result_T2 = mesolve(H, psi_T2, tlist, c_ops_T2, [a.dag()*a], options=opts)

# Plotting
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(tlist * 1e6, result_T1.expect[0])
plt.xlabel("Time (μs)")
plt.ylabel("Population in |1⟩")
plt.title("T₁ Relaxation")

plt.subplot(1, 2, 2)
plt.plot(tlist * 1e6, result_T2.expect[0].real)
plt.xlabel("Time (μs)")
plt.ylabel("Coherence ⟨1|ρ|1⟩")
plt.title("T₂ Dephasing")

plt.tight_layout()
plt.grid(True)
plt.show()

In [ ]:
plt.savefig("results/one_qubit_chip/relaxation_times.png", dpi=300, bbox_inches='tight')
plt.close()